# TarKG TF-GNNS Embedding Inspection

This notebook loads the TarKG model artifacts produced by `07_tarkg_sampler_engine_tfgnns_training.ipynb`, opens the GestaltDB source graph, and runs qualitative nearest-neighbor checks over learned KG node embeddings.

It checks whether nodes with the same semantic kind, especially drugs and diseases, tend to appear near each other in embedding space.

## 1. Setup

Run this notebook from the repo root or from `notebooks/`. Use the same Python environment/kernel used for notebook 07 so TensorFlow, `tf_gnns`, `polars`, and the optional RocksDB backend are available.

In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import numpy as np
import polars as pl
import tensorflow as tf

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

for path in (PROJECT_ROOT, PROJECT_ROOT / "src"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from example.gnn_kg.config import make_config
from example.gnn_kg.model import KGContrastiveModel
from gestaltdb.graphdb import GraphDB
from gestaltdb.sampling import SamplerSnapshot

print("Project root:", PROJECT_ROOT)
print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

I0000 00:00:1787683972.506641 1194768 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1787683972.536563 1194768 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1787683973.179405 1194768 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Project root: /home/charilaos/Workspace/gestaltdb
TensorFlow: 2.21.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 2. Load Config, Snapshot, And GraphDB

The sampler snapshot provides compact ID to external ID mappings. The graph DB provides node properties for inspecting nearest neighbors.

In [2]:
cfg = make_config()

# Match the model/training overrides used in notebook 07.
cfg.sampler.batch_size = 64
cfg.sampler.negatives_per_positive = 8
cfg.sampler.hop1_fanout = 12
cfg.sampler.hop2_fanout = 8
cfg.sampler.engine_negative_source = "context_relation_neighbors"
cfg.model.hidden_dim = 512
cfg.model.mp_steps = 2

required_paths = [
    cfg.paths.sampler_snapshot_path / "metadata.json",
    cfg.paths.node_map_path,
    cfg.paths.db_path,
]
missing = [path for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError("Missing required TarKG artifacts: " + ", ".join(str(path) for path in missing))

snapshot = SamplerSnapshot.load(cfg.paths.sampler_snapshot_path, mmap=True)
print(f"Snapshot: {snapshot.num_nodes:,} nodes, {snapshot.num_edges:,} edges, {snapshot.num_relations:,} relations")
print("Snapshot source_db metadata:", snapshot.metadata.get("source_db"))

try:
    graph = snapshot.open_source_graph() if snapshot.metadata.get("source_db") else GraphDB.open(cfg.paths.db_path)
except Exception as exc:
    print("Managed GraphDB.open failed; falling back to explicit PyRexStore + JSONSerializer.")
    print(type(exc).__name__ + ":", exc)
    from gestaltdb.kvstores import PyRexStore
    from gestaltdb.serializers import JSONSerializer

    graph = GraphDB(PyRexStore(path=str(cfg.paths.db_path)), JSONSerializer())

print("Graph manifest:", getattr(graph, "manifest", None))

Snapshot: 998,869 nodes, 14,079,259 edges, 14 relations
Snapshot source_db metadata: None
Managed GraphDB.open failed; falling back to explicit PyRexStore + JSONSerializer.
ValueError: missing GestaltDB manifest: /home/charilaos/Workspace/gestaltdb/data/tarkg_gnn/gestaltdb_tarkg_subset_rocksdb/gestaltdb_manifest.json. Migrate an existing DB with graph.save_manifest().
Graph manifest: {'format_version': 1, 'gestaltdb_version': '0.5.2', 'backend': {'name': 'pyrex', 'class': 'gestaltdb.kvstores.PyRexStore', 'layout_version': 1, 'options': {}}, 'serializer': {'name': 'json', 'class': 'gestaltdb.serializers.JSONSerializer', 'format_version': 1}, 'graph': {'node_count': None, 'edge_count': None, 'indexed_node_properties': [], 'indexed_edge_properties': []}, 'created_at': '2026-08-25T18:52:57Z'}


## 3. Load Model Weights Or Exported Embeddings

Notebook 07 saves weight files in `notebooks/model_hist*_*.weights.h5` and also exports `data/tarkg_gnn/artifacts/tfgnns_sampler_engine_embeddings.npz`. This cell prefers the newest weights file. If full Keras model restoration fails because `tf_gnns` layer names changed, it still reads the trained embedding matrices directly from the `.weights.h5` checkpoint. Set `WEIGHTS_PATH` manually if you want a specific checkpoint.

In [ ]:
WEIGHTS_PATH = None  # Example: PROJECT_ROOT / "notebooks" / "model_hist182780_1.00E-04.weights.h5"
EMBEDDINGS_PATH = cfg.paths.artifacts_dir / "tfgnns_sampler_engine_embeddings.npz"

def latest_weights_file() -> Path | None:
    candidates = list((PROJECT_ROOT / "notebooks").glob("model_hist*_*.weights.h5"))
    candidates += list((PROJECT_ROOT / "notebooks").glob("model.weights.h5"))
    if not candidates:
        return None
    return max(candidates, key=lambda path: path.stat().st_mtime)

def build_model_for_loading() -> KGContrastiveModel:
    model = KGContrastiveModel(
        cfg.model,
        num_nodes=snapshot.num_nodes,
        num_node_types=max(1, int(snapshot.metadata.get("node_type_count", 1))),
        num_relations=snapshot.num_relations,
    )
    graph_in = {
        "nodes": tf.constant([[0, max(0, int(snapshot.node_type_ids[0]))], [min(1, snapshot.num_nodes - 1), max(0, int(snapshot.node_type_ids[min(1, snapshot.num_nodes - 1)]))]], dtype=tf.float32),
        "edges": tf.constant([[0]], dtype=tf.float32),
        "senders": tf.constant([0], dtype=tf.int64),
        "receivers": tf.constant([1], dtype=tf.int64),
        "n_nodes": tf.constant([2], dtype=tf.int64),
        "n_edges": tf.constant([1], dtype=tf.int64),
        "n_graphs": tf.constant(1, dtype=tf.int64),
    }
    labels = {
        "positive_src": tf.constant([0], dtype=tf.int64),
        "positive_dst": tf.constant([1], dtype=tf.int64),
        "positive_rel": tf.constant([0], dtype=tf.int64),
        "negative_src": tf.constant([[1]], dtype=tf.int64),
        "negative_dst": tf.constant([[0]], dtype=tf.int64),
        "negative_rel": tf.constant([[0]], dtype=tf.int64),
    }
    _ = model(graph_in, labels, training=False)
    return model

def load_embeddings_from_weights_h5(path: Path) -> tuple[np.ndarray, np.ndarray]:
    import h5py

    with h5py.File(path, "r") as handle:
        node = np.asarray(handle["layers/embedding/vars/0"], dtype=np.float32)
        relation_score = np.asarray(handle["layers/embedding_3/vars/0"], dtype=np.float32)
    if node.shape[0] != snapshot.num_nodes:
        raise ValueError(f"checkpoint node embedding count {node.shape[0]:,} != snapshot nodes {snapshot.num_nodes:,}")
    if relation_score.shape[0] != snapshot.num_relations:
        raise ValueError(f"checkpoint relation embedding count {relation_score.shape[0]:,} != snapshot relations {snapshot.num_relations:,}")
    return node, relation_score

weights_path = Path(WEIGHTS_PATH) if WEIGHTS_PATH is not None else latest_weights_file()
model = None
embedding_source = None

if weights_path is not None and weights_path.exists():
    try:
        model = build_model_for_loading()
        model.load_weights(weights_path)
        node_embeddings = model.node_id_emb.get_weights()[0].astype(np.float32)
        relation_score_embeddings = model.score_rel.get_weights()[0].astype(np.float32)
        print("Loaded full Keras model weights.")
    except Exception as exc:
        print("Full Keras model weight load failed; loading trained lookup embeddings directly from checkpoint.")
        print(type(exc).__name__ + ":", exc)
        try:
            node_embeddings, relation_score_embeddings = load_embeddings_from_weights_h5(weights_path)
            embedding_source = weights_path
        except Exception as h5_exc:
            if not EMBEDDINGS_PATH.exists():
                raise
            print("Direct checkpoint embedding load failed; falling back to exported .npz embeddings.")
            print(type(h5_exc).__name__ + ":", h5_exc)
            exported = np.load(EMBEDDINGS_PATH)
            node_embeddings = exported["node_id_embeddings"].astype(np.float32)
            relation_score_embeddings = exported["relation_score_embeddings"].astype(np.float32)
            embedding_source = EMBEDDINGS_PATH
    else:
        embedding_source = weights_path
elif EMBEDDINGS_PATH.exists():
    exported = np.load(EMBEDDINGS_PATH)
    node_embeddings = exported["node_id_embeddings"].astype(np.float32)
    relation_score_embeddings = exported["relation_score_embeddings"].astype(np.float32)
    embedding_source = EMBEDDINGS_PATH
else:
    raise FileNotFoundError("No model weights or exported embedding archive found. Run notebook 07 first.")

print("Embedding source:", embedding_source)
print("Node embeddings:", node_embeddings.shape)
print("Relation score embeddings:", relation_score_embeddings.shape)

## 4. Build Inspection Tables

This joins the sampler compact IDs to the TarKG node map. If the parquet map is unavailable or incomplete, the helper falls back to GraphDB node properties.

In [ ]:
try:
    node_map = pl.read_parquet(cfg.paths.node_map_path).sort("node_int")
except Exception as exc:
    print("Could not read node_map with polars; using minimal snapshot-derived table.")
    print(type(exc).__name__ + ":", exc)
    node_map = pl.DataFrame({"node_int": np.arange(snapshot.num_nodes), "node_id": snapshot.external_node_ids.astype(str)})

if "node_int" not in node_map.columns:
    node_map = node_map.with_row_index("node_int")
if "node_id" not in node_map.columns:
    node_map = node_map.with_columns(pl.Series("node_id", snapshot.external_node_ids.astype(str)))
if "kind" not in node_map.columns:
    node_map = node_map.with_columns(pl.lit(None).alias("kind"))

snapshot_ids = pl.DataFrame({"node_int": np.arange(snapshot.num_nodes), "snapshot_node_id": snapshot.external_node_ids.astype(str)})
node_info = snapshot_ids.join(node_map, on="node_int", how="left").with_columns(
    pl.coalesce([pl.col("node_id"), pl.col("snapshot_node_id")]).alias("node_id"),
    pl.col("kind").fill_null("unknown").cast(pl.Utf8).alias("kind"),
)

node_ids = np.asarray(node_info["node_id"].to_list(), dtype=str)
node_kind_labels = np.asarray(node_info["kind"].to_list(), dtype=str)
node_kinds = np.char.lower(node_kind_labels)
kind_counts = node_info.group_by("kind").len().sort("len", descending=True).head(20)
display(kind_counts)
display(node_info.head())

## 5. Nearest Neighbor Helpers

Cosine similarity is computed against all nodes by default. For quick checks on very large embeddings, set `MAX_CANDIDATES` to sample candidate nodes.

In [ ]:
MAX_CANDIDATES = None  # Example: 200_000 for faster approximate checks
RANDOM_SEED = 13

normed_embeddings = node_embeddings / np.maximum(np.linalg.norm(node_embeddings, axis=1, keepdims=True), 1e-9)
rng = np.random.default_rng(RANDOM_SEED)
candidate_ids = np.arange(snapshot.num_nodes, dtype=np.int64)
if MAX_CANDIDATES is not None and MAX_CANDIDATES < snapshot.num_nodes:
    candidate_ids = np.sort(rng.choice(candidate_ids, size=MAX_CANDIDATES, replace=False))

def node_record(node_int: int):
    node_int = int(node_int)
    return {
        "node_int": node_int,
        "node_id": str(node_ids[node_int]),
        "kind": str(node_kind_labels[node_int]),
        "kind_key": str(node_kinds[node_int]),
    }

def graph_node_properties(node_id: str) -> dict:
    try:
        node = graph.get_node(node_id.encode("utf-8"))
    except Exception:
        return {}
    return {} if node is None else dict(node.properties)

def nearest_nodes(node_int: int, k: int = 10, kind: str | None = None) -> pl.DataFrame:
    node_int = int(node_int)
    candidates = candidate_ids
    if kind is not None:
        kind_mask = node_kinds[candidates] == kind.lower()
        candidates = candidates[kind_mask]
    candidates = candidates[candidates != node_int]
    if candidates.size == 0:
        return pl.DataFrame()
    scores = normed_embeddings[candidates] @ normed_embeddings[node_int]
    take = min(k, scores.size)
    order = np.argpartition(-scores, kth=take - 1)[:take]
    order = order[np.argsort(-scores[order])]
    rows = []
    for pos in order:
        neighbor_int = int(candidates[pos])
        record = node_record(neighbor_int)
        record["cosine"] = float(scores[pos])
        record["properties"] = str(graph_node_properties(record["node_id"]))
        rows.append(record)
    return pl.DataFrame(rows)

def inspect_node(node_int: int, k: int = 10, restrict_to_same_kind: bool = False):
    seed = node_record(node_int)
    kind = seed["kind_key"] if restrict_to_same_kind else None
    print("Seed:", seed)
    print("Seed GraphDB properties:", graph_node_properties(seed["node_id"]))
    return nearest_nodes(node_int, k=k, kind=kind)

## 6. Drug Similarity Checks

Sample several drug nodes and inspect their nearest neighbors. Good signs: nearest neighbors are often other drugs or biologically connected entities, and same-kind-restricted neighbors are plausible drug IDs.

In [ ]:
def sample_nodes_by_kind(kind: str, n: int = 5) -> np.ndarray:
    ids = node_info.filter(pl.col("kind").str.to_lowercase() == kind.lower())["node_int"].to_numpy().astype(np.int64)
    if ids.size == 0:
        print(f"No nodes found with kind={kind!r}")
        return ids
    return np.sort(rng.choice(ids, size=min(n, ids.size), replace=False))

drug_seed_ids = sample_nodes_by_kind("drug", n=5)
drug_seed_ids

In [ ]:
for node_int in drug_seed_ids:
    print("\n" + "=" * 100)
    display(inspect_node(int(node_int), k=10, restrict_to_same_kind=False))
    print("Same-kind drug neighbors")
    display(inspect_node(int(node_int), k=10, restrict_to_same_kind=True))

## 7. Disease Similarity Checks

Repeat the same nearest-neighbor check for disease nodes.

In [ ]:
disease_seed_ids = sample_nodes_by_kind("disease", n=5)
disease_seed_ids

In [ ]:
for node_int in disease_seed_ids:
    print("\n" + "=" * 100)
    display(inspect_node(int(node_int), k=10, restrict_to_same_kind=False))
    print("Same-kind disease neighbors")
    display(inspect_node(int(node_int), k=10, restrict_to_same_kind=True))

## 8. Aggregate Same-Kind Sanity Metrics

This computes how often the top-K nearest neighbors share the seed node kind for a random sample of drugs and diseases. It is not a formal evaluation metric, but it is useful for catching broken ID alignment or untrained/random embeddings.

In [ ]:
def same_kind_neighbor_rate(kind: str, sample_size: int = 100, k: int = 20) -> pl.DataFrame:
    ids = node_info.filter(pl.col("kind").str.to_lowercase() == kind.lower())["node_int"].to_numpy().astype(np.int64)
    if ids.size == 0:
        return pl.DataFrame()
    seeds = rng.choice(ids, size=min(sample_size, ids.size), replace=False)
    rows = []
    for seed in seeds:
        nn = nearest_nodes(int(seed), k=k)
        if nn.empty:
            continue
        rows.append({
            "seed_node_int": int(seed),
            "seed_node_id": node_record(int(seed))["node_id"],
            "kind": kind,
            "top_k": len(nn),
            "same_kind_count": int((nn["kind_key"] == kind.lower()).sum()),
            "same_kind_rate": float((nn["kind_key"] == kind.lower()).mean()),
            "mean_cosine": float(nn["cosine"].mean()),
        })
    return pl.DataFrame(rows)

drug_rates = same_kind_neighbor_rate("drug", sample_size=100, k=20)
disease_rates = same_kind_neighbor_rate("disease", sample_size=100, k=20)
summary = pl.concat([drug_rates, disease_rates], how="vertical") if drug_rates.height or disease_rates.height else pl.DataFrame()
display(summary.group_by("kind").agg(
    pl.mean("same_kind_rate").alias("same_kind_rate_mean"),
    pl.median("same_kind_rate").alias("same_kind_rate_median"),
    pl.std("same_kind_rate").alias("same_kind_rate_std"),
    pl.mean("same_kind_count").alias("same_kind_count_mean"),
    pl.mean("mean_cosine").alias("mean_cosine_mean"),
    pl.len().alias("count"),
))
display(summary.head(20))

## 9. Relation Embedding Quick Look

This is a small relation embedding nearest-neighbor check using the DistMult scoring relation embeddings.

In [ ]:
relation_ids = snapshot.external_relation_ids.astype(str)
rel_emb = relation_score_embeddings / np.maximum(np.linalg.norm(relation_score_embeddings, axis=1, keepdims=True), 1e-9)
rel_scores = rel_emb @ rel_emb.T

rows = []
for rel_idx, rel_name in enumerate(relation_ids):
    order = np.argsort(-rel_scores[rel_idx])
    neighbors = [(relation_ids[j], float(rel_scores[rel_idx, j])) for j in order if j != rel_idx][:5]
    rows.append({"relation": rel_name, "nearest_relations": neighbors})
display(pl.DataFrame(rows))

## 10. Cleanup

Close the DB handle when done.

In [ ]:
graph.close()